# Part 3: Many qubits

Practice notebook for *Qubit to Classifier*. Run the examples, fill in the exercises, and compare with the solutions at the end. Every cell needs only Python 3 and NumPy, except one optional Qiskit exercise in chapter 4.1.

## 3.1 Two qubits and the tensor product

### Combining qubits with `np.kron`

`np.kron` computes the tensor product: `np.kron(a, b)` puts a in q0 and b in q1. The amplitudes come out in the order 00, 01, 10, 11, with q0 as the first (leftmost) bit, the same order as this course. The helper `show` prints the nonzero amplitudes of a state, each with its label.

In [ ]:
import numpy as np

s = 1 / np.sqrt(2)
ket0, ket1 = np.array([1, 0]), np.array([0, 1])
plus, minus = np.array([s, s]), np.array([s, -s])


def show(psi):
    n = int(np.log2(len(psi)))
    for k, a in enumerate(psi):
        if abs(a) > 1e-12:
            print(f"   |{k:0{n}b}>  {a:+.4f}")


print("|+> (x) |0>:")
show(np.kron(plus, ket0))
print("|+> (x) |->:")
show(np.kron(plus, minus))
print("|+>|+>|+> has", len(np.kron(np.kron(plus, plus), plus)), "amplitudes, each", round(s ** 3, 4))

### Exercise 3.1.1: Probabilities for one qubit at a time

For |ψ⟩ = (1/2)|00⟩ + (1/2)|01⟩ + (1/√2)|11⟩, find P(q0 = 0) and P(q1 = 1) by reshaping the four probabilities into a 2 × 2 grid: rows for q0, columns for q1.

<details><summary>Hint</summary>

`probs.reshape(2, 2)` puts |00⟩, |01⟩ in the first row and |10⟩, |11⟩ in the second. Then add up a row or a column.

</details>

In [ ]:
# Your code here

### Exercise 3.1.2: A gate on one qubit of many

Write `on_qubit(gate, q, n)`, which builds the 2<sup>n</sup> × 2<sup>n</sup> matrix that applies a one-qubit gate to qubit q of n qubits and leaves the rest alone. Check the qubit order first: X on qubit 0 should turn |000⟩ into |100⟩. Then use it to apply H to each of three qubits, starting from |000⟩.

<details><summary>Hint</summary>

The matrix is a tensor product with `gate` in position q, counting from the left (q0 is the leftmost factor), and the 2 × 2 identity everywhere else: I ⊗ … ⊗ gate ⊗ … ⊗ I.

</details>

In [ ]:
# Your code here

### Exercise 3.1.3: How memory grows

Print how many bytes a full state vector needs for n = 10, 20, 30, 40 and 50 qubits, at 16 bytes per complex amplitude, in convenient units.

<details><summary>Hint</summary>

The number of amplitudes is 2<sup>n</sup> (`2 ** n` in Python). Divide by 1,000 repeatedly to step through kB, MB, GB, TB and PB.

</details>

In [ ]:
# Your code here

## 3.2 Multi-qubit gates

### Two-qubit gates as 4 × 4 matrices

On two qubits a state has four amplitudes (00, 01, 10, 11) and a gate is a 4 × 4 matrix. A one-qubit gate on q0 is `np.kron(gate, I)`. Here H and CNOT make the Bell state, and CZ entangles |+⟩|+⟩.

In [ ]:
import numpy as np

I = np.eye(2)
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
CNOT = np.array([[1, 0, 0, 0],      # control q0, target q1
                 [0, 1, 0, 0],
                 [0, 0, 0, 1],
                 [0, 0, 1, 0]])
CZ = np.diag([1, 1, 1, -1])
ket00 = np.array([1, 0, 0, 0])

bell = CNOT @ np.kron(H, I) @ ket00          # H on q0, then CNOT
print("H then CNOT on |00>:", bell.round(4))
plus_plus = np.kron(H, H) @ ket00
print("CZ on |+>|+>:       ", (CZ @ plus_plus).round(4))

### Exercise 3.2.1: Phase kickback

Apply CNOT to |+⟩|−⟩ and show that the result is |−⟩|−⟩: the target did not change, but the control did.

<details><summary>Hint</summary>

Build |+⟩|−⟩ and |−⟩|−⟩ with `np.kron` and compare with `np.allclose`.

</details>

In [ ]:
# Your code here

### Exercise 3.2.2: Build any controlled gate

Write `controlled(U)`, which returns |0⟩⟨0| ⊗ I + |1⟩⟨1| ⊗ U with q0 as the control. Check that `controlled(X)` is CNOT and `controlled(Z)` is CZ. Then compare controlled-Rz(π/2) with controlled-S: are they equal now?

<details><summary>Hint</summary>

|0⟩⟨0| is `np.diag([1, 0])` and |1⟩⟨1| is `np.diag([0, 1])`.

</details>

In [ ]:
# Your code here

## 3.3 Entanglement

### Is it entangled, and how much?

Three tools from the chapter in code: the product test α₀₀α₁₁ − α₀₁α₁₀, the Bloch vector of q0 on its own, and the entanglement entropy. The Bloch vector of q0 comes from the density matrix of q0 alone, found by a partial trace (chapter 6.1), which NumPy computes in one line.

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])


def product_test(psi):
    a = psi.reshape(2, 2)                    # a[q0, q1]
    return a[0, 0] * a[1, 1] - a[0, 1] * a[1, 0]


def bloch_q0(psi):
    a = psi.reshape(2, 2)
    rho = a @ a.conj().T                     # the state of q0 alone
    return np.array([np.trace(rho @ P).real for P in (X, Y, Z)])


def entropy(psi):
    length = np.linalg.norm(bloch_q0(psi))
    lam = np.clip((1 + length) / 2, 1e-12, 1 - 1e-12)
    return -lam * np.log2(lam) - (1 - lam) * np.log2(1 - lam)


states = {
    "|+>|+>":              np.array([1, 1, 1, 1]) / 2,
    "Bell (|00>+|11>)/√2": np.array([1, 0, 0, 1]) / np.sqrt(2),
    "(|00>+|01>+|11>)/√3": np.array([1, 1, 0, 1]) / np.sqrt(3),
}
for name, psi in states.items():
    print(f"{name:20s} test = {product_test(psi):.3f}   |r_q0| = {np.linalg.norm(bloch_q0(psi)):.3f}   S = {entropy(psi) + 0:.3f} bits")

The product state gives 0 in the product test, and its q0 arrow has length 1. The Bell state's q0 arrow has length 0, and its entropy is a full bit. The third state is partly entangled: the test gives 1/3, the arrow is shorter than 1 but not 0, and the entropy is about half a bit.

### Exercise 3.3.1: The entangler, Ry(θ) then CNOT

Prepare Ry(θ)|0⟩ on q0, then apply CNOT. For θ = 0, π/6, π/3 and π/2, print the length of q0's Bloch vector and the entanglement entropy. Check that the length is |cos θ|, and that θ = π/3 gives about 0.81 bits.

<details><summary>Hint</summary>

The state is cos(θ/2)|00⟩ + sin(θ/2)|11⟩, so you can also write it down directly instead of multiplying matrices.

</details>

In [ ]:
# Your code here

### Exercise 3.3.2: Beat the classical limit in the CHSH game

For the Bell state |Φ⁺⟩, compute the correlation E(α, β) = ⟨Φ⁺|A ⊗ B|Φ⁺⟩, where A = sin α X + cos α Z measures along the angle α in the x–z plane (and B likewise for β). Then compute S = E(a, b) − E(a, b′) + E(a′, b) + E(a′, b′) for a = 0, a′ = π/2, b = π/4 and b′ = 3π/4.

<details><summary>Hint</summary>

`np.kron(A, B)` is the two-qubit observable, and the correlation is `phi @ np.kron(A, B) @ phi` for the real state phi.

</details>

In [ ]:
# Your code here

### Exercise 3.3.3: Measure one qubit of a pair

For (|00⟩ + |01⟩ + |11⟩)/√3, compute P(q0 = 0). Then write down the state after q0 reads 0: keep only the labels that start with 0, and renormalize. Which state is q1 left in?

<details><summary>Hint</summary>

Set the amplitudes with q0 = 1 to zero, then divide by the length of what is left.

</details>

In [ ]:
# Your code here

---

# Solutions

### Solution 3.1.1: Probabilities for one qubit at a time

In [ ]:
import numpy as np

psi = np.array([1 / 2, 1 / 2, 0, 1 / np.sqrt(2)])     # order: 00, 01, 10, 11
grid = (np.abs(psi) ** 2).reshape(2, 2)              # grid[q0, q1]
print("grid of probabilities:\n", grid.round(3))
print("P(q0 = 0) =", grid[0, :].sum().round(3))      # first row
print("P(q1 = 1) =", grid[:, 1].sum().round(3))      # second column

Expected output:

```
grid of probabilities:
 [[0.25 0.25]
 [0.   0.5 ]]
P(q0 = 0) = 0.5
P(q1 = 1) = 0.75
```

The same reshaping trick works for any number of qubits: reshape to (2, 2, …, 2) and add up over the qubits you are not asking about.

### Solution 3.1.2: A gate on one qubit of many

In [ ]:
import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
X = np.array([[0, 1], [1, 0]])


def on_qubit(gate, q, n):
    m = np.array([[1.0]])
    for k in range(n):
        m = np.kron(m, gate if k == q else np.eye(2))
    return m


start = np.zeros(8)
start[0] = 1                                 # |000>
flipped = on_qubit(X, 0, 3) @ start
print(f"X on q0: |000> -> |{np.argmax(flipped):03b}>")

psi = start
for q in range(3):
    psi = on_qubit(H, q, 3) @ psi
print("amplitudes:", psi.round(4))

Expected output:

```
X on q0: |000> -> |100>
amplitudes: [0.3536 0.3536 0.3536 0.3536 0.3536 0.3536 0.3536 0.3536]
```

X on q0 gives |100⟩, index 4, because q0 is the leftmost bit; a matrix built in the reverse order would give |001⟩. Every amplitude ends up 1/(2√2) ≈ 0.3536: H on every qubit makes the uniform superposition of all eight basis states.

### Solution 3.1.3: How memory grows

In [ ]:
def readable(nbytes):
    for unit in ["bytes", "kB", "MB", "GB", "TB", "PB", "EB"]:
        if nbytes < 1000:
            return f"{nbytes:.0f} {unit}"
        nbytes /= 1000
    return f"{nbytes:.0f} ZB"


for n in [10, 20, 30, 40, 50]:
    print(f"{n} qubits: {2 ** n:>20,} amplitudes, {readable(16 * 2 ** n)}")

Expected output:

```
10 qubits:                1,024 amplitudes, 16 kB
20 qubits:            1,048,576 amplitudes, 17 MB
30 qubits:        1,073,741,824 amplitudes, 17 GB
40 qubits:    1,099,511,627,776 amplitudes, 18 TB
50 qubits: 1,125,899,906,842,624 amplitudes, 18 PB
```

### Solution 3.2.1: Phase kickback

In [ ]:
import numpy as np

s = 1 / np.sqrt(2)
plus, minus = np.array([s, s]), np.array([s, -s])
CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])

out = CNOT @ np.kron(plus, minus)
print("CNOT |+>|-> =", out.round(4))
print("equal to |->|->:", np.allclose(out, np.kron(minus, minus)))

Expected output:

```
CNOT |+>|-> = [ 0.5 -0.5 -0.5  0.5]
equal to |->|->: True
```

### Solution 3.2.2: Build any controlled gate

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Z = np.diag([1, -1])
S = np.diag([1, 1j])
Rz = np.diag([np.exp(-1j * np.pi / 4), np.exp(1j * np.pi / 4)])   # Rz(pi/2)


def controlled(U):
    return np.kron(np.diag([1, 0]), np.eye(2)) + np.kron(np.diag([0, 1]), U)


CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])
print("controlled(X) is CNOT:", np.allclose(controlled(X), CNOT))
print("controlled(Z) is CZ:  ", np.allclose(controlled(Z), np.diag([1, 1, 1, -1])))
print("controlled(Rz(pi/2)) equals controlled(S):", np.allclose(controlled(Rz), controlled(S)))
print("diagonal of controlled(Rz(pi/2)):", np.diag(controlled(Rz)).round(4))

Expected output:

```
controlled(X) is CNOT: True
controlled(Z) is CZ:   True
controlled(Rz(pi/2)) equals controlled(S): False
diagonal of controlled(Rz(pi/2)): [1.    +0.j     1.    +0.j     0.7071-0.7071j 0.7071+0.7071j]
```

S and Rz(π/2) differ only by a global phase, but once they are controlled that phase becomes a relative phase on the control, and the two gates differ.

### Solution 3.3.1: The entangler, Ry(θ) then CNOT

In [ ]:
import numpy as np

CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])


def ry(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s], [s, c]])


def q0_length_and_entropy(psi):
    a = psi.reshape(2, 2)
    rho = a @ a.conj().T
    lam = np.linalg.eigvalsh(rho)                  # the two eigenvalues of q0's state
    entropy = -sum(x * np.log2(x) for x in lam if x > 1e-12)
    length = abs(lam[1] - lam[0])                  # Bloch length = difference of eigenvalues
    return length, entropy


for theta in [0, np.pi / 6, np.pi / 3, np.pi / 2]:
    psi = CNOT @ np.kron(ry(theta) @ [1, 0], [1, 0])
    length, S = q0_length_and_entropy(psi)
    print(f"theta = {theta:.4f}:  |r| = {length:.3f}  |cos theta| = {abs(np.cos(theta)):.3f}  S = {S + 0:.3f} bits")

Expected output:

```
theta = 0.0000:  |r| = 1.000  |cos theta| = 1.000  S = 0.000 bits
theta = 0.5236:  |r| = 0.866  |cos theta| = 0.866  S = 0.355 bits
theta = 1.0472:  |r| = 0.500  |cos theta| = 0.500  S = 0.811 bits
theta = 1.5708:  |r| = 0.000  |cos theta| = 0.000  S = 1.000 bits
```

### Solution 3.3.2: Beat the classical limit in the CHSH game

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])
phi_plus = np.array([1, 0, 0, 1]) / np.sqrt(2)


def axis(angle):
    return np.sin(angle) * X + np.cos(angle) * Z


def E(alpha, beta):
    return phi_plus @ np.kron(axis(alpha), axis(beta)) @ phi_plus


a, a2, b, b2 = 0, np.pi / 2, np.pi / 4, 3 * np.pi / 4
S = E(a, b) - E(a, b2) + E(a2, b) + E(a2, b2)
print(f"E(a,b) = {E(a, b):.3f}, E(a,b') = {E(a, b2):.3f}, E(a',b) = {E(a2, b):.3f}, E(a',b') = {E(a2, b2):.3f}")
print(f"S = {S:.4f}    2*sqrt(2) = {2 * np.sqrt(2):.4f}")

Expected output:

```
E(a,b) = 0.707, E(a,b') = -0.707, E(a',b) = 0.707, E(a',b') = 0.707
S = 2.8284    2*sqrt(2) = 2.8284
```

S = 2√2 ≈ 2.828 is above 2, the largest value that answers fixed in advance can reach.

### Solution 3.3.3: Measure one qubit of a pair

In [ ]:
import numpy as np

psi = np.array([1, 1, 0, 1]) / np.sqrt(3)           # order: 00, 01, 10, 11
p_q0_is_0 = np.sum(np.abs(psi[:2]) ** 2)             # labels 00 and 01
after = psi.copy()
after[2:] = 0                                        # cross out 10 and 11
after = after / np.linalg.norm(after)                # renormalize
print(f"P(q0 = 0) = {p_q0_is_0:.4f}")
print("state after reading 0:", after.round(4))
print("q1 alone is", after[:2].round(4), "which is |+>")

Expected output:

```
P(q0 = 0) = 0.6667
state after reading 0: [0.7071 0.7071 0.     0.    ]
q1 alone is [0.7071 0.7071] which is |+>
```